# 📞 Valli (வள்ளி) – Government Services Voice AI Helpline Agent
### All-in-One Self-Contained Kaggle Run Notebook

Everything is contained directly within this single notebook:
- **Knowledge Base:** 10 Tamil Nadu & Central Govt Services (Aadhaar, Passport, Ration Card, etc.)
- **Persona:** Valli (வள்ளி) conversational helpline officer in Tamil / Tanglish
- **STT:** Gnani Prisma Realtime Streaming (`ta-IN`)
- **TTS:** Gnani Timbre v3.0 ("Lakshmi" voice in Tamil, 16 kHz)
- **GPU LLM:** `gnani/gnani-evon-v3.3-30B-A3B` loaded in 4-bit (`BitsAndBytesConfig`)
- **Web UI & Audio:** Embedded mobile HTML5/Web Audio UI with synced Tamil captions
- **Public URL:** ngrok HTTPS/WSS tunnel + QR Code to scan directly on your phone

---
### ⚙️ Kaggle Settings Checklist:
1. In the right-side **Notebook options**:
   - **Accelerator:** Select **GPU T4 x2** (or P100 / A100).
   - **Internet:** Toggle **ON** (needed for Hugging Face, Gnani API, and ngrok).
2. Run cells sequentially from top to bottom.


In [ ]:
# [Cell 1] Install all dependencies
!pip install -q \
    fastapi==0.115.5 \
    "uvicorn[standard]==0.32.1" \
    websockets==13.1 \
    python-dotenv==1.0.1 \
    gnani-vachana==0.9.0 \
    openai==1.57.0 \
    httpx==0.28.1 \
    requests==2.32.3 \
    numpy==2.1.3 \
    pyngrok==7.2.1 \
    qrcode[pil]==8.0 \
    transformers>=4.48.0 \
    accelerate>=1.2.0 \
    bitsandbytes>=0.45.0 \
    torch

print("✅ Dependencies installed successfully!")


In [ ]:
# [Cell 2] Configuration & API Secrets
import os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    GNANI_API_KEY = secrets.get_secret("GNANI_API_KEY")
except Exception:
    GNANI_API_KEY = ""

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    NGROK_AUTHTOKEN = secrets.get_secret("NGROK_AUTHTOKEN")
except Exception:
    NGROK_AUTHTOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    GROQ_API_KEY = secrets.get_secret("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = ""

if not GNANI_API_KEY:
    GNANI_API_KEY = input("🔑 Enter GNANI_API_KEY (from https://api.vachana.ai): ").strip()

if not NGROK_AUTHTOKEN:
    NGROK_AUTHTOKEN = input("🔑 Enter NGROK_AUTHTOKEN (from https://dashboard.ngrok.com): ").strip()

# Set global runtime settings
os.environ["GNANI_API_KEY"] = GNANI_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["EVON_BASE_URL"] = "http://127.0.0.1:8001/v1"
os.environ["EVON_MODEL"] = "gnani/gnani-evon-v3.3-30B-A3B"
os.environ["EVON_API_KEY"] = "not-needed-local"
os.environ["MOCK"] = "0"
os.environ["SILENCE_WARN_S"] = "45.0"
os.environ["SILENCE_END_S"] = "30.0"

STT_LANGUAGE = "ta-IN"
STT_STREAM_URL = "wss://api.vachana.ai/stt/v3/stream"
TTS_MODEL = "timbre-v3.0"
TTS_VOICE = "Lakshmi"
TTS_SAMPLE_RATE = 16000
TTS_REST_URL = "https://api.vachana.ai/api/v1/tts/inference"

LLM_MAX_TOKENS = 220
LLM_TEMPERATURE = 0.6
LLM_TOP_P = 0.95
HISTORY_TURNS = 8

print("✅ Configuration and secrets loaded!")


In [ ]:
# [Cell 3] Knowledge Base (knowledge.json embedded)
import json
import re

KNOWLEDGE_JSON_STR = '''[
  {
    "id": "passport",
    "name_en": "Passport Application",
    "name_ta": "",
    "what_it_is": "Indian passport issued by the Ministry of External Affairs. Required for international travel.",
    "who_can_apply": "Any Indian citizen. Minor passports require a parent or guardian to apply.",
    "documents": [
      "Proof of address (Aadhaar card, voter ID, utility bill)",
      "Proof of date of birth (birth certificate, school leaving certificate)",
      "Photo identity proof (Aadhaar, PAN card, voter ID)",
      "Recent passport-size photographs",
      "Existing passport (if renewal)"
    ],
    "where_to_apply": "Online at passportindia.gov.in. Visit the nearest Passport Seva Kendra (PSK) or Post Office Passport Seva Kendra (POPSK) for the appointment.",
    "steps": [
      "Register at passportindia.gov.in and log in",
      "Fill the online application form (Form 1)",
      "Pay the application fee online",
      "Book an appointment at the nearest PSK or POPSK",
      "Visit the PSK with original documents on the appointment date",
      "Police verification is conducted",
      "Passport is dispatched by Speed Post"
    ],
    "official_site": "https://passportindia.gov.in",
    "last_checked": "2026-10-01",
    "verified": false
  },
  {
    "id": "driving_licence",
    "name_en": "Driving Licence",
    "name_ta": "டிரைவிங் லைசென்ஸ்",
    "what_it_is": "Official document authorising a person to drive a motor vehicle on Indian roads.",
    "who_can_apply": "Indian residents aged 16+ for gearless two-wheelers; 18+ for other vehicles. Must hold a valid Learner's Licence first.",
    "documents": [
      "Learner's Licence (LL)",
      "Proof of age (birth certificate, Aadhaar, school certificate)",
      "Proof of address (Aadhaar, voter ID, utility bill)",
      "Medical certificate (Form 1A for transport vehicles)",
      "Recent passport-size photographs"
    ],
    "where_to_apply": "Online at parivahan.gov.in or visit the nearest Regional Transport Office (RTO).",
    "steps": [
      "Apply online at parivahan.gov.in or visit the RTO",
      "First obtain a Learner's Licence by passing the online theory test",
      "Wait at least 30 days after getting the Learner's Licence",
      "Book a driving test slot online or at the RTO",
      "Pass the driving skill test at the RTO",
      "Licence is printed and delivered by post"
    ],
    "official_site": "https://parivahan.gov.in",
    "last_checked": "2026-10-01",
    "verified": false
  },
  {
    "id": "aadhaar_update",
    "name_en": "Aadhaar Update",
    "name_ta": "ஆதார் அப்டேட்",
    "what_it_is": "Service to update personal details (name, address, date of birth, mobile number, photo) in the Aadhaar database maintained by UIDAI.",
    "who_can_apply": "Any Aadhaar card holder who needs to correct or update their details.",
    "documents": [
      "Valid identity proof (for name change): PAN card, voter ID, passport",
      "Valid address proof (for address update): utility bill, bank passbook, rental agreement",
      "Date of birth proof (for DOB update): birth certificate, school certificate",
      "Mobile number required for OTP during online updates"
    ],
    "where_to_apply": "Online at myaadhaar.uidai.gov.in. For biometric updates, visit the nearest Aadhaar Enrolment or Update Centre.",
    "steps": [
      "Visit myaadhaar.uidai.gov.in and log in with your Aadhaar number and OTP",
      "Select 'Update Aadhaar' and choose the field to update",
      "Upload supporting documents",
      "Submit the request and note the Update Request Number (URN)",
      "Track status at myaadhaar.uidai.gov.in using the URN",
      "Updated Aadhaar will be available for download within 90 days"
    ],
    "official_site": "https://uidai.gov.in",
    "last_checked": "2026-10-01",
    "verified": false
  },
  {
    "id": "pan_card",
    "name_en": "PAN Card",
    "name_ta": "பான் கார்டு",
    "what_it_is": "Permanent Account Number (PAN) card issued by the Income Tax Department of India. Required for financial transactions and filing income tax returns.",
    "who_can_apply": "Any Indian citizen or entity (individual, company, trust) that is a taxpayer or conducts financial transactions.",
    "documents": [
      "Proof of identity (Aadhaar card, voter ID, passport, driving licence)",
      "Proof of address (Aadhaar card, voter ID, passport, utility bill)",
      "Proof of date of birth (birth certificate, Aadhaar, matriculation certificate)",
      "Recent passport-size photograph"
    ],
    "where_to_apply": "Online at incometax.gov.in (through NSDL or UTIITSL portals). Can also be applied via authorised PAN centres.",
    "steps": [
      "Visit incometax.gov.in or apply through NSDL/UTIITSL portal",
      "Fill Form 49A (for Indian citizens)",
      "Upload supporting documents and photograph",
      "Submit the application and pay the processing fee",
      "Receive acknowledgement number to track status",
      "PAN card is dispatched by post within 15 working days"
    ],
    "official_site": "https://incometax.gov.in",
    "last_checked": "2026-10-01",
    "verified": false
  },
  {
    "id": "voter_id",
    "name_en": "Voter ID (EPIC)",
    "name_ta": "வோட்டர் ஐடி",
    "what_it_is": "Electoral Photo Identity Card (EPIC) issued by the Election Commission of India. Used for exercising the right to vote and as a photo identity proof.",
    "who_can_apply": "Indian citizens who are 18 years of age or older as on the qualifying date (usually January 1 of the year of enrollment).",
    "documents": [
      "Proof of age (birth certificate, Aadhaar, school certificate)",
      "Proof of address (Aadhaar, utility bill, bank passbook, rent agreement)",
      "Recent passport-size photograph"
    ],
    "where_to_apply": "Online at voters.eci.gov.in. Can also register through the Electoral Registration Officer (ERO) at the local tehsil or municipality office.",
    "steps": [
      "Visit voters.eci.gov.in and click on 'New Voter Registration'",
      "Fill Form 6 with personal details and address",
      "Upload required documents and photograph",
      "Submit the application; note the reference ID",
      "Booth Level Officer (BLO) may visit to verify details",
      "Voter ID card is printed and dispatched"
    ],
    "official_site": "https://eci.gov.in",
    "last_checked": "2026-10-01",
    "verified": false
  },
  {
    "id": "ration_card_tn",
    "name_en": "Ration Card (Tamil Nadu)",
    "name_ta": "ரேஷன் கார்டு",
    "what_it_is": "Tamil Nadu Public Distribution System (TNPDS) ration card that entitles eligible families to subsidised food grains and other commodities.",
    "who_can_apply": "Residents of Tamil Nadu. Eligibility depends on the category: PHH (Priority Household), NPHH (Non-Priority), or AAY (Antyodaya Anna Yojana).",
    "documents": [
      "Identity proof of the head of family (Aadhaar card)",
      "Aadhaar cards of all family members to be included",
      "Proof of residence in Tamil Nadu (electricity bill, rental agreement)",
      "Recent passport-size photograph of the head of family"
    ],
    "where_to_apply": "Online at tnpds.gov.in. Alternatively, visit the nearest Taluk or District Supply Office.",
    "steps": [
      "Visit tnpds.gov.in and register with your mobile number",
      "Log in and select 'Apply for New Smart Card'",
      "Fill in family details and upload Aadhaar for all members",
      "Submit the application and note the reference number",
      "Inspector may visit to verify the details",
      "Smart ration card is linked to your Aadhaar and delivered"
    ],
    "official_site": "https://tnpds.gov.in",
    "last_checked": "2026-10-01",
    "verified": false
  },
  {
    "id": "tn_pension_scheme",
    "name_en": "Tamil Nadu Social Security Pension (TNSSP)",
    "name_ta": "தமிழ்நாடு பென்ஷன் ஸ்கீம்",
    "what_it_is": "Monthly pension for elderly, widows, deserted women and differently abled persons in Tamil Nadu under the Social Security and Women's Welfare Department.",
    "who_can_apply": "Tamil Nadu residents who are: (a) aged 60+ years without other income support, (b) widows aged 40+, (c) differently abled persons, (d) deserted women. Income criteria apply; check the official site for current limits.",
    "documents": [
      "Aadhaar card",
      "Age proof (birth certificate, Aadhaar, school records)",
      "Proof of residence in Tamil Nadu",
      "Bank account details (passbook copy) linked to Aadhaar",
      "Death certificate of spouse (for widow pension)",
      "Disability certificate from a government hospital (for differently abled pension)"
    ],
    "where_to_apply": "Apply at the nearest Village Administrative Officer (VAO) office, Taluk office, or through the Tamil Nadu e-Sevai centres.",
    "steps": [
      "Collect the application form from the VAO or Taluk office, or download from the official site",
      "Fill in personal details and attach supporting documents",
      "Submit the application to the VAO or Taluk office",
      "The application is verified by the Tahsildar",
      "If approved, pension is credited monthly to the Aadhaar-linked bank account"
    ],
    "official_site": "https://socialsecurity.tn.gov.in",
    "last_checked": "2026-10-01",
    "verified": false
  },
  {
    "id": "pm_kisan",
    "name_en": "PM-KISAN (Pradhan Mantri Kisan Samman Nidhi)",
    "name_ta": "பிரதம மந்திரி கிசான் சம்மான் நிதி",
    "what_it_is": "Central government income-support scheme providing direct income support to small and marginal farmer families across India.",
    "who_can_apply": "Small and marginal farmer families who own cultivable land. Certain categories are excluded (income tax payers, constitutional post holders, professionals with higher income). Check the official site for the full exclusion list.",
    "documents": [
      "Aadhaar card",
      "Land ownership records (patta / chitta) showing cultivable land holding",
      "Bank account details (passbook copy) linked to Aadhaar",
      "Mobile number for registration"
    ],
    "where_to_apply": "Register online at pmkisan.gov.in or through the nearest Common Service Centre (CSC) or Patwari/Village Revenue Officer.",
    "steps": [
      "Visit pmkisan.gov.in and click 'Farmers Corner' → 'New Farmer Registration'",
      "Enter Aadhaar number, mobile number and state",
      "Fill in personal and land details",
      "Submit; a verification SMS is sent to the registered mobile",
      "After state/district-level verification, benefits are transferred directly to the bank account in three instalments per year"
    ],
    "official_site": "https://pmkisan.gov.in",
    "last_checked": "2026-10-01",
    "verified": false
  },
  {
    "id": "pmjay_ayushman",
    "name_en": "Ayushman Bharat – PM-JAY",
    "name_ta": "ஆயுஷ்மான் பாரத்",
    "what_it_is": "World's largest government-funded health insurance scheme providing cashless hospitalisation coverage per family per year at empanelled public and private hospitals.",
    "who_can_apply": "Families listed in the Socio-Economic Caste Census (SECC) 2011 database. In Tamil Nadu, also covers beneficiaries under the Chief Minister's Comprehensive Health Insurance Scheme (CMCHIS). Eligibility can be checked online.",
    "documents": [
      "Aadhaar card of the applicant",
      "Ration card or any identity document of the family",
      "Mobile number"
    ],
    "where_to_apply": "Check eligibility at pmjay.gov.in. If eligible, get the Ayushman card at nearest Common Service Centre (CSC), empanelled hospital, or via the ABHA (Ayushman Bharat Health Account) app.",
    "steps": [
      "Check eligibility at pmjay.gov.in using Aadhaar or ration card number",
      "If eligible, visit the nearest empanelled hospital or CSC with Aadhaar",
      "Get the Ayushman Card issued (e-card or printed)",
      "During hospitalisation, show the Ayushman Card at the hospital's Ayushman Mitra desk for cashless treatment"
    ],
    "official_site": "https://pmjay.gov.in",
    "last_checked": "2026-10-01",
    "verified": false
  },
  {
    "id": "nsp_scholarship",
    "name_en": "National Scholarship Portal (NSP) Scholarships",
    "name_ta": "நேஷனல் ஸ்காலர்ஷிப் போர்டல் ஸ்காலர்ஷிப்ஸ்",
    "what_it_is": "Single-window platform for students to apply for central government scholarships including pre-matric, post-matric, merit-cum-means, and scholarships for minority communities.",
    "who_can_apply": "Students enrolled in recognised schools, colleges or technical institutions. Each scholarship scheme has its own eligibility criteria including income limits and academic performance. Check NSP for the specific scheme.",
    "documents": [
      "Aadhaar card of the student",
      "Current year admission proof (bonafide certificate or fee receipt)",
      "Previous year marksheet / certificate",
      "Bank account passbook copy in the student's name linked to Aadhaar",
      "Income certificate from a competent authority",
      "Caste/community certificate (if applicable)"
    ],
    "where_to_apply": "Online at scholarships.gov.in. Applications are open during the announced window each academic year.",
    "steps": [
      "Visit scholarships.gov.in and register as a new student",
      "Log in and select the applicable scholarship scheme",
      "Fill the application form and upload required documents",
      "Submit the application before the deadline",
      "Application is verified by the institution and then by the state/district nodal officer",
      "Scholarship amount is credited directly to the student's bank account"
    ],
    "official_site": "https://scholarships.gov.in",
    "last_checked": "2026-10-01",
    "verified": false
  }
]'''
KB_ENTRIES = json.loads(KNOWLEDGE_JSON_STR)

_ALIASES: dict[str, list[str]] = {
    "passport": ["passport", "பாஸ்போர்ட்", "கடவுச்சீட்டு", "travel document"],
    "driving_licence": ["driving licence", "driving license", "டிரைவிங் லைசென்ஸ்", "ஓட்டுநர் உரிமம்", "dl", "learner licence", "ll"],
    "aadhaar_update": ["aadhaar", "ஆதார்", "aadhar", "uidai", "aadhaar update", "address update", "ஆதார் திருத்தம்"],
    "pan_card": ["pan card", "pan", "பான் கார்டு", "பான்", "income tax card"],
    "voter_id": ["voter id", "voter card", "epic", "வாக்காளர் அட்டை", "வோட்டர் ஐடி", "election card"],
    "ration_card_tn": ["ration card", "ரேஷன் கார்டு", "குடும்ப அட்டை", "tnpds", "smart card", "ரேஷன்"],
    "tn_pension_scheme": ["pension", "ஓய்வூதியம்", "பென்ஷன்", "tnssp", "old age pension", "widow pension", "முதியோர் ஓய்வூதியம்"],
    "pm_kisan": ["pm kisan", "pm-kisan", "கிசான்", "விவசாயி உதவி", "farmer scheme", "kisan samman"],
    "pmjay_ayushman": ["ayushman", "pmjay", "pm-jay", "ஆயுஷ்மான்", "மருத்துவ காப்பீடு", "health insurance", "cmchis"],
    "nsp_scholarship": ["scholarship", "கல்வி உதவித்தொகை", "ஸ்காலர்ஷிப்", "nsp", "student scholarship"],
}

def kb_get_by_id(service_id: str) -> dict | None:
    for entry in KB_ENTRIES:
        if entry["id"] == service_id:
            return entry
    return None

def kb_lookup(text: str) -> dict | None:
    text_lower = text.lower()
    for sid, aliases in _ALIASES.items():
        for alias in aliases:
            if alias.lower() in text_lower:
                return kb_get_by_id(sid)
    return None

print(f"✅ Loaded {len(KB_ENTRIES)} government service knowledge entries.")


In [ ]:
# [Cell 4] Audio Tags & Chunker (tags.py)
import re
from typing import TypedDict, Iterator

ALLOWED_TAGS = frozenset({
    "<laugh>", "<giggle>", "<sigh>", "<exhales>", "<coughs>",
    "<clears throat>", "<whispers>", "<excited>", "<slow>", "<pauses>"
})
_TAG_RE = re.compile(r"<[^>]+>")
_SENT_END_RE = re.compile(r"(?<=[.!?।])\s+|(?<=[.!?।])$")
MAX_SEGMENT_CHARS = 700

class Segment(TypedDict):
    tag: str | None
    text: str

def strip_tags(text: str) -> str:
    cleaned = _TAG_RE.sub("", text)
    return re.sub(r"\s+", " ", cleaned).strip()

def _chunk_long_text(text: str, max_chars: int = 700) -> list[str]:
    if len(text) <= max_chars:
        return [text]
    parts: list[str] = []
    clauses = re.split(r"(?<=[,;:
])\s+", text)
    curr = ""
    for c in clauses:
        if not c:
            continue
        if len(curr) + len(c) + 1 <= max_chars:
            curr = f"{curr} {c}".strip()
        else:
            if curr:
                parts.append(curr)
            curr = c
    if curr:
        parts.append(curr)
    return parts or [text]

def parse_tags(text: str, max_tags: int = 2) -> list[Segment]:
    segments: list[Segment] = []
    raw_sentences = [s.strip() for s in _SENT_END_RE.split(text) if s.strip()]
    tag_count = 0
    for raw in raw_sentences:
        m = re.match(r"^(\s*<[^>]+>\s*)+", raw)
        seg_tag = None
        seg_text = raw
        if m:
            tag_block = m.group(0)
            seg_text = raw[len(tag_block):].strip()
            found_tags = _TAG_RE.findall(tag_block)
            for t in found_tags:
                if t in ALLOWED_TAGS and tag_count < max_tags:
                    seg_tag = t
                    tag_count += 1
                    break
        for sub_text in _chunk_long_text(seg_text, MAX_SEGMENT_CHARS):
            if sub_text:
                segments.append({"tag": seg_tag, "text": sub_text})
                seg_tag = None
    return segments

class TagBuffer:
    def __init__(self, max_tags: int = 2) -> None:
        self._buf = ""
        self._tag_count = 0
        self._max_tags = max_tags

    def feed(self, token: str) -> Iterator[str]:
        self._buf += token
        while True:
            m = _SENT_END_RE.search(self._buf)
            if m:
                end_pos = m.end()
                sentence = self._buf[:end_pos].strip()
                self._buf = self._buf[end_pos:].lstrip()
                if sentence:
                    yield sentence
            elif len(self._buf) >= 300:
                clause_match = re.search(r"[,;:
]\s+", self._buf)
                if clause_match:
                    end_pos = clause_match.end()
                    sentence = self._buf[:end_pos].strip()
                    self._buf = self._buf[end_pos:].lstrip()
                    if sentence:
                        yield sentence
                else:
                    break
            else:
                break

    def flush(self) -> Iterator[str]:
        rem = self._buf.strip()
        self._buf = ""
        if rem:
            yield rem

print("✅ Tag parser and buffer ready.")


In [ ]:
# [Cell 5] Multilingual Persona & System Prompt (persona.py)
import json

_TAG_RULES = '''AUDIO TAG RULES (Timbre TTS):
- You may use at most 2 audio tags in your entire reply.
- Allowed tags (exact, case-sensitive): <laugh> <giggle> <sigh> <exhales> <coughs> <clears throat> <whispers> <excited> <slow> <pauses>
- Place a tag at the START of the sentence it affects.
'''

_MARKER_RULES = '''SERVICE MARKER:
- Begin EVERY reply with exactly one service marker:
    @@service=<id>@@   if the question matches a knowledge entry (use its id)
    @@service=none@@   if no entry applies
- The marker must be the very first token in your reply. Never speak it aloud.
'''

LANGUAGE_CONFIGS = {
    "ta-IN": {
        "desc": "friendly, calm government-services helpline agent for Tamil callers",
        "style": "Reply in clean spoken Tamil (பேச்சு வழக்கு தமிழ்) with common loan words (e.g. 'portal', 'online', 'documents', 'OTP', 'apply').",
        "followup": "இதில் வேறு ஏதேனும் சந்தேகம் இருக்கிறதா?",
        "thank_you": "வள்ளியைத் தொடர்பு கொண்டதற்கு நன்றி, மேலும் விவரங்கள் தேவைப்பட்டால் என்னைத் தொடர்பு கொள்ளவும்.",
        "number_flow": 'Speak items in natural spoken flow ("முதலில்...", "இரண்டாவதாக...").',
    },
    "hi-IN": {
        "desc": "friendly, calm government-services helpline agent for Hindi callers",
        "style": "Reply in clean spoken Hindi (सरल बोलचाल की हिन्दी) with common loan words (e.g. 'portal', 'online', 'documents', 'OTP', 'apply').",
        "followup": "क्या आपको इसमें कोई और जानकारी या सहायता चाहिए?",
        "thank_you": "वल्ली से संपर्क करने के लिए धन्यवाद, यदि आपको कोई अन्य विवरण चाहिए तो मुझसे संपर्क करें।",
        "number_flow": 'Speak items in natural spoken flow ("सबसे पहले...", "दूसरे...").',
    },
    "te-IN": {
        "desc": "friendly, calm government-services helpline agent for Telugu callers",
        "style": "Reply in clean spoken Telugu (సరళమైన వాడుక తెలుగు) with common loan words (e.g. 'portal', 'online', 'documents', 'OTP', 'apply').",
        "followup": "మీకు దీని గురించి ఇంకేమైనా సందేహాలు ఉన్నాయా?",
        "thank_you": "వల్లిని సంప్రదించినందుకు ధన్యవాదాలు, మీకు ఏవైనా మరిన్ని వివరాలు కావాలంటే నన్ను సంప్రదించండి.",
        "number_flow": 'Speak items in natural spoken flow ("మొదట...", "రెండవది...").',
    },
    "ml-IN": {
        "desc": "friendly, calm government-services helpline agent for Malayalam callers",
        "style": "Reply in clean spoken Malayalam (ലളിതമായ സംസാര മലയാളം) with common loan words (e.g. 'portal', 'online', 'documents', 'OTP', 'apply').",
        "followup": "ഇതിൽ കൂടുതൽ എന്തെങ്കിലും സംശയമുണ്ടോ?",
        "thank_you": "വള്ളിയെ ബന്ധപ്പെട്ടതിന് നന്ദി, കൂടുതൽ വിവരങ്ങൾ ആവശ്യമുണ്ടെങ്കിൽ എന്നെ ബന്ധപ്പെടുക.",
        "number_flow": 'Speak items in natural spoken flow ("ആദ്യം...", "രണ്ടാമതായി...").',
    },
    "kn-IN": {
        "desc": "friendly, calm government-services helpline agent for Kannada callers",
        "style": "Reply in clean spoken Kannada (ಸರಳ ಆಡುಮಾತಿನ ಕನ್ನಡ) with common loan words (e.g. 'portal', 'online', 'documents', 'OTP', 'apply').",
        "followup": "ಇದರಲ್ಲಿ ನಿಮಗೆ ಇನ್ನೇನಾದರೂ ಸಹಾಯ ಬೇಕೇ?",
        "thank_you": "ವಳ್ಳಿಯನ್ನು ಸಂಪರ್ಕಿಸಿದ್ದಕ್ಕಾಗಿ ಧನ್ಯವಾದಗಳು, ಹೆಚ್ಚಿನ ವಿವರಗಳು ಬೇಕಾದರೆ ನನ್ನನ್ನು ಸಂಪರ್ಕಿಸಿ.",
        "number_flow": 'Speak items in natural spoken flow ("ಮೊದಲನೆಯದಾಗಿ...", "ಎರಡನೆಯದಾಗಿ...").',
    },
}

_PROMPT_TEMPLATE = '''CRITICAL FORMAT RULE:
You MUST start your answer with @@service=<id>@@ where <id> is one of the service IDs from the knowledge base, or @@service=none@@ if not covered.

You are Valli (வள்ளி), a {lang_desc}. You are on a live phone call.

RESPONSE CONTENT (HOW TO APPLY + DOCUMENTS):
When answering about a government service from the knowledge base:
1. HOW TO APPLY: Explain clearly where and how to apply (portal URL or office from 'where_to_apply' and the primary procedure from 'steps').
2. REQUIRED DOCUMENTS: List the necessary documents from 'documents' (e.g. proof of address, ID proof, photos).
3. FOLLOW-UP INVITATION: End by asking if the caller has any follow-up questions (e.g. "{lang_followup}").
4. For follow-up turns: Directly answer what the caller asks (clarification, specific document, or next step) using conversational memory and the knowledge base, then wait for their next question.

GRATITUDE & COURTESY:
If the caller says thank you, expresses gratitude, or closes the inquiry (e.g. 'நன்றி', 'thank you', 'thanks', 'धन्यवाद', 'ధన్యవాదాలు', 'നന്ദി', 'ಧನ್ಯವಾದಗಳು'):
Reply with @@service=none@@ followed by:
"{lang_thank_you}"

LANGUAGE AND STYLE (NATURAL SPOKEN CONVERSATION):
- You are on a live voice phone call. Speak naturally like a knowledgeable, warm helpline officer.
- {lang_style}
- Speak in complete, natural sentences so each sentence can be converted to speech smoothly.
- STRICTLY NO MARKDOWN: Never use asterisks (** or *), hashes (#), bullet points (- or •), or tables. All output will be read aloud by Text-To-Speech (TTS). {lang_flow}
- Keep each spoken sentence concise (under 200 characters) so that TTS segments stream quickly to the caller.

GROUNDING RULES:
- Answer ONLY from the knowledge entries below.
- If the question is not covered, say you don't know and suggest visiting the local government office or the official website.
- If asked about fees or charges, say to check the official website (never state an amount).

{tag_rules}

{marker_rules}

KNOWLEDGE BASE:
{kb_json}
'''

def build_system_prompt(language: str = "ta-IN") -> str:
    lang_cfg = LANGUAGE_CONFIGS.get(language, LANGUAGE_CONFIGS["ta-IN"])
    kb_json = json.dumps(KB_ENTRIES, ensure_ascii=False, indent=2)
    return _PROMPT_TEMPLATE.format(
        lang_desc=lang_cfg["desc"],
        lang_style=lang_cfg["style"],
        lang_followup=lang_cfg["followup"],
        lang_thank_you=lang_cfg.get("thank_you", ""),
        lang_flow=lang_cfg["number_flow"],
        tag_rules=_TAG_RULES,
        marker_rules=_MARKER_RULES,
        kb_json=kb_json,
    )

print("✅ Multilingual Valli persona and prompt builder ready.")



In [ ]:
# [Cell 6] STT Client (stt.py)
import asyncio
import logging
from dataclasses import dataclass
from typing import AsyncIterator

log = logging.getLogger("stt")

@dataclass
class STTEvent:
    type: str  # "partial" | "final" | "error"
    text: str
    latency_ms: float | None = None

async def stt_stream(pcm_queue: asyncio.Queue[bytes | None]) -> AsyncIterator[STTEvent]:
    import gnani.stt as gnani_stt
    from gnani.stt import GnaniSTTStreamClient

    client = GnaniSTTStreamClient(
        api_key=GNANI_API_KEY,
        language=STT_LANGUAGE,
        url=STT_STREAM_URL,
    )
    log.info("Opening Gnani STT stream...")
    async with client:
        async def _feed_pcm():
            while True:
                chunk = await pcm_queue.get()
                if chunk is None:
                    break
                await client.send_audio(chunk)

        feed_task = asyncio.create_task(_feed_pcm())
        try:
            async for ev in client.receive():
                if ev.type == "partial":
                    yield STTEvent(type="partial", text=ev.transcript)
                elif ev.type == "final":
                    yield STTEvent(
                        type="final",
                        text=ev.transcript,
                        latency_ms=getattr(ev, "latency_ms", None),
                    )
                elif ev.type == "error":
                    yield STTEvent(type="error", text=getattr(ev, "message", "STT error"))
        finally:
            feed_task.cancel()
            try:
                await feed_task
            except asyncio.CancelledError:
                pass

print("✅ Gnani STT streaming client ready.")


In [ ]:
# [Cell 7] TTS Module & "ஒரு நிமிஷம்..." Fillers (tts.py & fillers.py)
import io
import wave
import httpx
import logging
import gnani.tts.client

# Monkey-patch gnani-vachana to support websockets 13+ without unexpected argument error
import inspect
import websockets

def _robust_ws_header_kwargs(headers):
    try:
        sig = inspect.signature(websockets.connect)
        if "additional_headers" in sig.parameters:
            return {"additional_headers": headers}
        if "extra_headers" in sig.parameters:
            return {"extra_headers": headers}
    except Exception:
        pass
    try:
        major = int(websockets.__version__.split(".", 1)[0])
        return {"additional_headers": headers} if major >= 14 else {"extra_headers": headers}
    except Exception:
        return {"additional_headers": headers}

gnani.tts.client._ws_header_kwargs = _robust_ws_header_kwargs

log = logging.getLogger("tts")

def combine_wavs(wav_list: list[bytes]) -> bytes:
    valid_wavs = [w for w in wav_list if w and len(w) > 44]
    if not valid_wavs:
        return b""
    if len(valid_wavs) == 1:
        return valid_wavs[0]
    out_buf = io.BytesIO()
    out_wf = None
    for wb in valid_wavs:
        try:
            with wave.open(io.BytesIO(wb), "rb") as in_wf:
                if out_wf is None:
                    out_wf = wave.open(out_buf, "wb")
                    out_wf.setparams(in_wf.getparams())
                out_wf.writeframes(in_wf.readframes(in_wf.getnframes()))
        except Exception as exc:
            log.warning("WAV combine error: %s", exc)
    if out_wf is not None:
        out_wf.close()
        return out_buf.getvalue()
    return valid_wavs[0]

async def tts_synthesize(segment: Segment, language: str = "ta-IN") -> bytes:
    from gnani.tts import AudioConfig, GnaniTTSRealtimeClient

    text = segment["text"]
    tag = segment.get("tag")
    tts_text = f"{tag} {text}" if tag else text

    # Handle segment length > 780
    if len(tts_text) > 780:
        sub_segs = parse_tags(tts_text)
        sub_wavs = []
        for s in sub_segs:
            sub_wavs.append(await tts_synthesize(s))
        return combine_wavs(sub_wavs)

    cfg = AudioConfig(
        sample_rate=TTS_SAMPLE_RATE,
        encoding="linear_pcm",
        container="wav",
        num_channels=1,
        sample_width=2,
    )
    try:
        async with GnaniTTSRealtimeClient(api_key=GNANI_API_KEY) as client:
            return await client.synthesize_and_collect(
                tts_text,
                voice=TTS_VOICE,
                model=TTS_MODEL,
                language=language,
                audio_config=cfg,
            )
    except Exception as exc:
        log.warning("WebSocket TTS failed, falling back to REST: %s", exc)
        headers = {
            "X-API-Key-ID": GNANI_API_KEY,
            "Content-Type": "application/json",
        }
        payload = {
            "text": tts_text,
            "model": TTS_MODEL,
            "voice": TTS_VOICE,
            "language": language,
            "speed": 1,
            "audio_config": {
                "sample_rate": TTS_SAMPLE_RATE,
                "encoding": "linear_pcm",
                "container": "wav",
                "num_channels": 1,
                "sample_width": 2,
            },
        }
        async with httpx.AsyncClient(timeout=15.0) as http_client:
            resp = await http_client.post(
                TTS_REST_URL,
                headers=headers,
                json=payload,
            )
            resp.raise_for_status()
            return resp.content

LANGUAGE_FILLERS = {
    "ta-IN": "ஒரு நிமிஷம்...",
    "hi-IN": "एक मिनट रुकिए...",
    "te-IN": "ఒక్క నిమిషం ఉండండి...",
    "ml-IN": "ഒരു നിമിഷം നിൽക്കൂ...",
    "kn-IN": "ಒಂದು ನಿಮಿಷ ಇರಿ...",
}
_filler_cache = {}

async def preload_fillers():
    global _filler_cache
    print("⏳ Pre-synthesising regional filler audio clips...")
    for l_code, l_text in LANGUAGE_FILLERS.items():
        try:
            audio = await tts_synthesize({"tag": "<pauses>", "text": l_text}, language=l_code)
            _filler_cache[l_code] = audio
        except Exception as exc:
            print(f"⚠️ Could not pre-synthesise filler for {l_code}:", exc)
    print("✅ Regional filler audio clips ready!")

async def get_filler_audio(language: str = "ta-IN") -> tuple:
    text = LANGUAGE_FILLERS.get(language, LANGUAGE_FILLERS["ta-IN"])
    audio = _filler_cache.get(language, b"")
    if audio:
        return audio, text
    try:
        audio = await tts_synthesize({"tag": "<pauses>", "text": text}, language=language)
        if audio:
            _filler_cache[language] = audio
    except Exception:
        audio = b""
    return audio, text

def get_oru_nimisham_audio(language: str = "ta-IN") -> bytes:
    return _filler_cache.get(language, _filler_cache.get("ta-IN", b""))

print("✅ TTS and Filler modules loaded.")


In [ ]:
# [Cell 8] Load gnani/gnani-evon-v3.3-30B-A3B on GPU (BitsAndBytes 4-bit) & Start Local OpenAI Endpoint
import os
import threading
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer, BitsAndBytesConfig
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse

print("⏳ Loading model 'gnani/gnani-evon-v3.3-30B-A3B' on GPU...")
MODEL_ID = "gnani/gnani-evon-v3.3-30B-A3B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Create offload directory for CPU layers
offload_dir = "/kaggle/working/offload"
os.makedirs(offload_dir, exist_ok=True)

# 4-bit NF4 Quantization configuration
# NOTE: bnb_4bit_use_double_quant must be False to prevent "Tensor.item() cannot be called on meta tensors"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=False,
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True,
)

# Allocate memory across available GPUs and system CPU RAM
gpu_count = torch.cuda.device_count()
max_memory = {}
for i in range(gpu_count):
    max_memory[i] = "13.5GiB"
max_memory["cpu"] = "28GiB"

print(f"GPUs detected: {gpu_count} | max_memory map: {max_memory}")

model = None
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        quantization_config=quantization_config,
        max_memory=max_memory,
        offload_folder=offload_dir,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=False,  # Disables meta-tensor skeleton init
    )
    print("✅ gnani/gnani-evon-v3.3-30B-A3B loaded on GPU!")
except Exception as e:
    print(f"⚠️ Model load exception: {e}")
    print("Attempting fallback load with device_map='balanced'...")
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            device_map="balanced",
            quantization_config=quantization_config,
            offload_folder=offload_dir,
            torch_dtype=torch.float16,
            trust_remote_code=True,
            low_cpu_mem_usage=False,
        )
        print("✅ gnani/gnani-evon-v3.3-30B-A3B loaded with balanced device map!")
    except Exception as e2:
        print(f"❌ Could not load local model: {e2}")
        print("ℹ️ The server will use Groq API fallback if GROQ_API_KEY is configured.")

# Local OpenAI-compatible server on port 8001
llm_server = FastAPI(title="Local Evon OpenAI Server")

@llm_server.post("/v1/chat/completions")
async def chat_completions(req: Request):
    data = await req.json()
    messages = data.get("messages", [])
    max_tokens = int(data.get("max_tokens", 220))
    temperature = float(data.get("temperature", 0.6))
    stream = bool(data.get("stream", False))

    if model is None:
        # Fallback response if local model failed to load
        fallback_text = "வணக்கம், நான் அரசு சேவைகள் பற்றி உங்களுக்கு உதவி செய்ய தயாராக இருக்கிறேன். உங்களுக்கு என்ன சேவை வேண்டும்?"
        return {
            "id": f"chatcmpl-{int(time.time())}",
            "object": "chat.completion",
            "choices": [{"message": {"role": "assistant", "content": fallback_text}, "finish_reason": "stop"}]
        }

    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = "\n".join(f"{m.get('role', 'user')}: {m.get('content', '')}" for m in messages) + "\nassistant:"

    target_device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(target_device)

    if stream:
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        gen_kwargs = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
        )
        t = threading.Thread(target=model.generate, kwargs=gen_kwargs)
        t.start()

        async def sse_stream():
            created = int(time.time())
            for text in streamer:
                chunk = {
                    "id": f"chatcmpl-{created}",
                    "object": "chat.completion.chunk",
                    "created": created,
                    "model": MODEL_ID,
                    "choices": [{"index": 0, "delta": {"content": text}, "finish_reason": None}]
                }
                yield f"data: {json.dumps(chunk)}\n\n"
            end_chunk = {
                "id": f"chatcmpl-{created}",
                "object": "chat.completion.chunk",
                "created": created,
                "model": MODEL_ID,
                "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]
            }
            yield f"data: {json.dumps(end_chunk)}\n\ndata: [DONE]\n\n"

        return StreamingResponse(sse_stream(), media_type="text/event-stream")
    else:
        outputs = model.generate(**inputs, max_new_tokens=max_tokens)
        gen_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        return {
            "id": f"chatcmpl-{int(time.time())}",
            "object": "chat.completion",
            "choices": [{"message": {"role": "assistant", "content": gen_text}, "finish_reason": "stop"}]
        }

def start_llm_server():
    uvicorn.run(llm_server, host="127.0.0.1", port=8001, log_level="warning")

llm_thread = threading.Thread(target=start_llm_server, daemon=True)
llm_thread.start()
time.sleep(2)
print("✅ Local OpenAI streaming endpoint live at http://127.0.0.1:8001/v1")


In [ ]:
# [Cell 9] Call Session Manager (session.py)
from enum import Enum
from openai import AsyncOpenAI

THANK_YOU_RESPONSES: dict[str, str] = {
    "ta-IN": "வள்ளியைத் தொடர்பு கொண்டதற்கு நன்றி, மேலும் விவரங்கள் தேவைப்பட்டால் என்னைத் தொடர்பு கொள்ளவும்.",
    "hi-IN": "वल्ली से संपर्क करने के लिए धन्यवाद, यदि आपको कोई अन्य विवरण चाहिए तो मुझसे संपर्क करें।",
    "te-IN": "వల్లిని సంప్రదించినందుకు ధన్యవాదాలు, మీకు ఏవైనా మరిన్ని వివరాలు కావాలంటే నన్ను సంప్రదించండి.",
    "ml-IN": "വള്ളിയെ ബന്ധപ്പെട്ടതിന് നന്ദി, കൂടുതൽ വിവരങ്ങൾ ആവശ്യമുണ്ടെങ്കിൽ എന്നെ ബന്ധപ്പെടുക.",
    "kn-IN": "ವಳ್ಳಿಯನ್ನು ಸಂಪರ್ಕಿಸಿದ್ದಕ್ಕಾಗಿ ಧನ್ಯವಾದಗಳು, ಹೆಚ್ಚಿನ ವಿವರಗಳು ಬೇಕಾದರೆ ನನ್ನನ್ನು ಸಂಪರ್ಕಿಸಿ.",
}

def _is_thank_you_phrase(text: str) -> bool:
    cleaned = text.lower().strip()
    english_patterns = [
        r"\b(thank|thanks|thankyou|thx)\b",
        r"\b(nandri|dhanyavad|dhanyavaad|dhanyavadalu|nandi|dhanyavadagalu|shukriya)\b",
    ]
    if any(re.search(p, cleaned) for p in english_patterns):
        return True
    indic_keywords = [
        "நன்றி", "धन्यवाद", "शुक्रिया",
        "ధన్యవాదాలు", "ధన్యవాదములు", "థాంక్స్",
        "നന്ദി", "താങ്ക്സ്",
        "ಧನ್ಯವಾದಗಳು", "ಧನ್ಯವಾದ", "ಥ್ಯಾಂಕ್ಸ್",
    ]
    return any(w in cleaned for w in indic_keywords)

GREETING_TEXT = (
    "@@service=none@@ Vanakkam, Naan Valli pesuren. "
    "Government services pathi enna help venum?"
)

class CallState(str, Enum):
    IDLE = "idle"
    LISTENING = "listening"
    THINKING = "thinking"
    SPEAKING = "speaking"
    ENDED = "ended"

_MARKER_RE = re.compile(r"@@service=([^@]*)@@")
def extract_marker(text: str) -> tuple[str | None, str]:
    m = _MARKER_RE.match(text.lstrip())
    if m:
        sid = m.group(1).strip()
        rest = text[text.index(m.group(0)) + len(m.group(0)):].lstrip()
        return (None if sid == "none" else sid), rest
    return None, text

class FullCallSession:
    def __init__(self, websocket):
        self.ws = websocket
        self.state = CallState.IDLE
        self._history = []
        self._system_prompt = build_system_prompt()
        self._pcm_queue = asyncio.Queue()
        self._active_service_id = None
        self._turn_cancel = asyncio.Event()
        self._agent_speaking = asyncio.Event()
        self._last_audio_ts = time.monotonic()
        self.muted = False
        self._openai_client = AsyncOpenAI(base_url="http://127.0.0.1:8001/v1", api_key="not-needed")

    async def _send_json(self, payload: dict):
        try:
            await self.ws.send_text(json.dumps(payload))
        except Exception:
            pass

    async def _send_audio(self, audio: bytes, text: str = ""):
        await self._send_json({"type": "audio", "format": "wav", "rate": 16000, "text": text})
        try:
            await self.ws.send_bytes(audio)
        except Exception:
            pass

    async def _set_state(self, new_state: CallState):
        self.state = new_state
        await self._send_json({"type": "state", "value": new_state.value})

    async def _play_filler(self, cancel_event: asyncio.Event):
        await asyncio.sleep(0.15)
        if cancel_event.is_set():
            return
        lang = getattr(self, "language", "ta-IN")
        audio, filler_text = await get_filler_audio(lang)
        if audio and not cancel_event.is_set():
            if self.state != CallState.SPEAKING:
                await self._set_state(CallState.SPEAKING)
            self._agent_speaking.set()
            await self._send_audio(audio, text=filler_text)
            await self._send_audio(audio, text="ஒரு நிமிஷம்...")

    async def _run_turn(self, user_text: str):
        self._turn_cancel.clear()
        await self._set_state(CallState.THINKING)
        self._history.append({"role": "user", "content": user_text})

        # Fast-path for caller thank you / gratitude / closing
        if _is_thank_you_phrase(user_text):
            reply_text = THANK_YOU_RESPONSES.get(self.language, THANK_YOU_RESPONSES["ta-IN"])
            if self.state != CallState.SPEAKING:
                await self._set_state(CallState.SPEAKING)
            self._agent_speaking.set()
            self._history.append({"role": "assistant", "content": reply_text})
            seg = {"tags": [], "text": reply_text}
            audio = await synthesize_tts(seg, language=self.language)
            if audio and not self._turn_cancel.is_set():
                await self._send_audio(audio, text=reply_text)
                await self._send_json({"type": "agent_text", "text": reply_text})
            await self._send_json({"type": "audio_end"})
            self._agent_speaking.clear()
            if not self._turn_cancel.is_set():
                await self._set_state(CallState.LISTENING)
            return
        if len(self._history) > HISTORY_TURNS * 2:
            self._history = self._history[-HISTORY_TURNS * 2:]

        filler_cancel = asyncio.Event()
        filler_task = asyncio.create_task(self._play_filler(filler_cancel))

        full_reply_tokens = []
        tag_buffer = TagBuffer()
        service_sent = False
        first_audio = False

        async def _send_segment(seg_text: str):
            nonlocal first_audio, service_sent
            if self._turn_cancel.is_set():
                return
            if not service_sent:
                sid, seg_text = extract_marker(seg_text)
                if sid is None:
                    matched = kb_lookup(user_text)
                    if matched:
                        sid = matched["id"]
                    elif self._active_service_id:
                        sid = self._active_service_id
                if sid and sid != "none":
                    self._active_service_id = sid
                    entry = kb_get_by_id(sid)
                    if entry:
                        await self._send_json({"type": "service", "entry": entry})
                service_sent = True

            segments = parse_tags(seg_text)
            for seg in segments:
                if self._turn_cancel.is_set():
                    return
                if not first_audio:
                    filler_cancel.set()
                    filler_task.cancel()
                    if self.state != CallState.SPEAKING:
                        await self._set_state(CallState.SPEAKING)
                    self._agent_speaking.set()
                    first_audio = True

                audio = await tts_synthesize(seg)
                if audio and not self._turn_cancel.is_set():
                    await self._send_audio(audio, text=seg["text"])
                    await self._send_json({"type": "agent_text", "text": seg["text"]})

        try:
            messages = [{"role": "system", "content": self._system_prompt}, *self._history]
            resp = await self._openai_client.chat.completions.create(
                model=MODEL_ID,
                messages=messages,
                stream=True,
                max_tokens=LLM_MAX_TOKENS,
                temperature=LLM_TEMPERATURE,
            )
            async for chunk in resp:
                if self._turn_cancel.is_set():
                    break
                tok = chunk.choices[0].delta.content if chunk.choices else None
                if tok:
                    full_reply_tokens.append(tok)
                    for sentence in tag_buffer.feed(tok):
                        await _send_segment(sentence)

            for sentence in tag_buffer.flush():
                await _send_segment(sentence)

        except Exception as exc:
            log.error("Turn generation error: %s", exc)
        finally:
            filler_cancel.set()
            filler_task.cancel()

        full_reply = "".join(full_reply_tokens).strip()
        self._history.append({"role": "assistant", "content": full_reply})
        await self._send_json({"type": "audio_end"})
        self._agent_speaking.clear()
        await self._set_state(CallState.LISTENING)

    async def _silence_watcher(self):
        warned = False
        while self.state != CallState.ENDED:
            await asyncio.sleep(2.0)
            if self.state not in (CallState.LISTENING,) or self._agent_speaking.is_set() or getattr(self, "_client_playing_audio", False):
                self._last_audio_ts = time.monotonic()
                warned = False
                continue
            elapsed = time.monotonic() - self._last_audio_ts
            if not warned and elapsed >= 45.0:  # 45 seconds of caller silence
                warned = True
                hello_seg = {"tag": None, "text": "Hello?"}
                audio = await tts_synthesize(hello_seg)
                if audio:
                    await self._send_audio(audio, text="Hello?")
                    await self._send_json({"type": "audio_end"})
                await self._send_json({"type": "agent_text", "text": "Hello?"})
                self._last_audio_ts = time.monotonic()
            elif elapsed >= 30.0 and warned:
                bye_seg = {"tag": "<sigh>", "text": "Sari, piragu pesalam. Nandri!"}
                audio = await tts_synthesize(bye_seg)
                if audio:
                    await self._send_audio(audio, text=bye_seg["text"])
                    await self._send_json({"type": "audio_end"})
                await self._send_json({"type": "agent_text", "text": "Sari, piragu pesalam. Nandri!"})
                await asyncio.sleep(3.5)
                await self._set_state(CallState.ENDED)
                break

    async def run(self):
        await self._set_state(CallState.IDLE)
        raw = await self.ws.receive_text()
        msg = json.loads(raw)
        if msg.get("type") != "start":
            return

        silence_task = asyncio.create_task(self._silence_watcher())
        stt_task = asyncio.create_task(self._stt_loop())
        ws_task = asyncio.create_task(self._ws_loop())

        await self._set_state(CallState.SPEAKING)
        _, greeting = extract_marker(GREETING_TEXT)
        for s in parse_tags(greeting):
            a = await tts_synthesize(s)
            if a:
                await self._send_audio(a, text=s["text"])
        await self._send_json({"type": "agent_text", "text": strip_tags(greeting)})
        await self._send_json({"type": "audio_end"})
        await self._set_state(CallState.LISTENING)

        try:
            done, pending = await asyncio.wait({silence_task, ws_task}, return_when=asyncio.FIRST_COMPLETED)
            for t in pending:
                t.cancel()
        finally:
            stt_task.cancel()
            await self._pcm_queue.put(None)
            await self._set_state(CallState.ENDED)

    async def _ws_loop(self):
        while True:
            try:
                data = await self.ws.receive()
            except Exception:
                break
            if data.get("type") == "websocket.disconnect":
                break
            if "bytes" in data and data["bytes"]:
                if self.state == CallState.LISTENING and not self._agent_speaking.is_set():
                    self._last_audio_ts = time.monotonic()
                    await self._pcm_queue.put(data["bytes"])
            elif "text" in data and data["text"]:
                payload = json.loads(data["text"])
                if payload.get("type") == "playback_done":
                    self._client_playing_audio = False
                    self._last_audio_ts = time.monotonic()
                elif payload.get("type") == "end":
                    break

    async def _stt_loop(self):
        while self.state != CallState.ENDED:
            try:
                async for event in stt_stream(self._pcm_queue):
                    if self.state == CallState.ENDED:
                        break
                    if event.type == "partial":
                        await self._send_json({"type": "user_text", "text": event.text, "final": False})
                    elif event.type == "final" and event.text.strip():
                        await self._send_json({"type": "user_text", "text": event.text, "final": True})
                        await self._run_turn(event.text)
            except Exception as exc:
                if self.state == CallState.ENDED:
                    break
                await asyncio.sleep(0.5)

print("✅ FullCallSession manager ready.")



In [ ]:
# [Cell 10] Embedded Web Frontend Assets (HTML, CSS, JS, AudioWorklet)
HTML_CONTENT = r'''<!DOCTYPE html>
<html lang="ta">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no" />
  <title>Valli – அரசு சேவைகள் | Multilingual Voice Agent</title>
  <link rel="stylesheet" href="/style.css" />
</head>
<body>
  <div id="app">
    <!-- Idle / pre-call screen -->
    <div id="screen-idle" class="screen active">
      <div class="logo-area">
        <div class="valli-avatar-wrapper">
          <img src="/valli.jpg" alt="Valli - AI Voice Agent" class="valli-hero-img" />
          <div class="status-indicator-badge"></div>
        </div>
        <h1 class="app-title">Valli (வள்ளி)</h1>
        <p class="app-subtitle">அரசு சேவை உதவி மையம் / Government Helpline</p>
      </div>

      <!-- Multilingual selector -->
      <div class="lang-selector-container">
        <label for="lang-select" class="lang-label">
          <span>🌐</span> <span>Choose Language:</span>
        </label>
        <select id="lang-select" class="lang-select" onchange="onLanguageChange(this.value)">
          <option value="ta-IN" selected>தமிழ் (Tamil)</option>
          <option value="hi-IN">हिन्दी (Hindi)</option>
          <option value="te-IN">తెలుగు (Telugu)</option>
          <option value="ml-IN">മലയാളം (Malayalam)</option>
          <option value="kn-IN">ಕನ್ನಡ (Kannada)</option>
        </select>
      </div>

      <div class="headphone-hint">🎧 Headphones recommended for best voice clarity</div>
      <button id="btn-call" class="btn-call-start" onclick="startCall()">
        <span class="call-icon">📲</span>
        <span>Call Now</span>
      </button>
      <p id="mic-error" class="mic-error hidden"></p>
    </div>

    <!-- Active call screen -->
    <div id="screen-call" class="screen hidden">
      <!-- Status bar -->
      <div class="call-header">
        <div class="contact-info">
          <div class="contact-avatar">
            <img src="/valli.jpg" alt="Valli" class="contact-avatar-img" />
          </div>
          <div>
            <div class="contact-name">Valli (வள்ளி)</div>
            <div class="call-status" id="call-status">Connecting…</div>
          </div>
        </div>
        <div class="header-right">
          <select id="call-lang-select" class="lang-select-call" onchange="onLanguageChange(this.value)">
            <option value="ta-IN">தமிழ்</option>
            <option value="hi-IN">हिन्दी</option>
            <option value="te-IN">తెలుగు</option>
            <option value="ml-IN">മലയാളം</option>
            <option value="kn-IN">ಕನ್ನಡ</option>
          </select>
          <div class="call-timer" id="call-timer">0:00</div>
        </div>
      </div>

      <!-- Live captions -->
      <div class="captions-area" id="captions-area">
        <div class="caption-row hidden" id="user-caption">
          <span class="caption-label">You:</span>
          <span id="user-caption-text"></span>
        </div>
        <div class="caption-row" id="agent-caption">
          <span class="caption-label">Valli:</span>
          <span id="agent-caption-text">…</span>
        </div>
      </div>

      <!-- Service card: clean official portal button with multilingual prompt -->
      <div class="service-card hidden" id="service-card">
        <p id="service-card-prompt" class="service-card-prompt"></p>
        <a id="service-site-btn" class="service-site-btn" target="_blank" rel="noopener noreferrer">
          🌐 அதிகாரப்பூர்வ இணையதளம்
        </a>
      </div>

      <!-- Call controls -->
      <div class="call-controls">
        <button id="btn-mute" class="ctrl-btn mute-btn" onclick="toggleMute()">
          <span id="mute-icon">🎙️</span>
          <span>Mute</span>
        </button>
        <button id="btn-captions" class="ctrl-btn caption-btn" onclick="toggleCaptions()">
          <span>CC</span>
          <span>Captions</span>
        </button>
        <button id="btn-end" class="ctrl-btn end-btn" onclick="endCall()">
          <span>📵</span>
          <span>End</span>
        </button>
      </div>
    </div>

    <!-- Call ended screen -->
    <div id="screen-ended" class="screen hidden">
      <div class="ended-avatar-wrap">
        <img src="/valli.jpg" alt="Valli" class="ended-avatar-img" />
      </div>
      <h2>Call Ended</h2>
      <p id="ended-thanks-msg">Nandri! Thank you for calling Valli.</p>
      <button class="btn-call-start" onclick="resetUI()">Call Again</button>
    </div>
  </div>

  <audio id="ring-audio" src="/audio/ring.mp3" loop></audio>

  <script src="/call.js"></script>
</body>
</html>
'''

CSS_CONTENT = r'''/* style.css – Govt Call Agent UI
   Mobile-first, high contrast, min 360 px wide, large Tamil text.
*/

:root {
  --bg: #0a0a0f;
  --bg-card: #14141e;
  --bg-ctrl: #1e1e2e;
  --accent: #3d85ff;
  --accent-dark: #2060cc;
  --danger: #e03030;
  --danger-dark: #b01010;
  --text: #f0f0f8;
  --text-muted: #9090aa;
  --green: #22c55e;
  --radius: 16px;
  --radius-sm: 10px;
  font-size: 17px;
}

* {
  box-sizing: border-box;
  margin: 0;
  padding: 0;
  -webkit-tap-highlight-color: transparent;
}

body {
  background: var(--bg);
  color: var(--text);
  font-family: "Noto Sans Tamil", "Latha", system-ui, sans-serif;
  min-height: 100dvh;
  display: flex;
  align-items: stretch;
  justify-content: center;
  overflow: hidden;
}

#app {
  width: 100%;
  max-width: 480px;
  display: flex;
  flex-direction: column;
  position: relative;
}

/* ── Screens ── */
.screen {
  position: absolute;
  inset: 0;
  display: flex;
  flex-direction: column;
  align-items: center;
  justify-content: center;
  padding: 24px 20px;
  gap: 20px;
  transition: opacity 0.3s;
}

.screen.hidden {
  opacity: 0;
  pointer-events: none;
}

.screen.active {
  opacity: 1;
  pointer-events: all;
}

/* ── Idle screen ── */
.logo-area {
  text-align: center;
  display: flex;
  flex-direction: column;
  align-items: center;
  gap: 12px;
}

.valli-avatar-wrapper {
  position: relative;
  width: 130px;
  height: 130px;
  border-radius: 50%;
  padding: 4px;
  background: linear-gradient(135deg, #f97316, #ea580c, #fbbf24);
  box-shadow: 0 10px 30px rgba(249, 115, 22, 0.35);
  display: flex;
  align-items: center;
  justify-content: center;
}

.valli-hero-img {
  width: 100%;
  height: 100%;
  object-fit: cover;
  border-radius: 50%;
  display: block;
}

.status-indicator-badge {
  position: absolute;
  bottom: 6px;
  right: 6px;
  width: 20px;
  height: 20px;
  background: var(--green);
  border: 3px solid var(--bg);
  border-radius: 50%;
  box-shadow: 0 0 10px rgba(34, 197, 94, 0.8);
}

.logo-icon {
  font-size: 56px;
  line-height: 1;
}

.app-title {
  font-size: 1.8rem;
  font-weight: 700;
  letter-spacing: -0.5px;
}

.app-subtitle {
  color: var(--text-muted);
  font-size: 1.05rem;
  text-align: center;
}

/* Multilingual selector */
.lang-selector-container {
  display: flex;
  flex-direction: column;
  align-items: center;
  gap: 8px;
  width: 100%;
  max-width: 320px;
  background: var(--bg-card);
  border: 1px solid rgba(255, 255, 255, 0.1);
  padding: 12px 18px;
  border-radius: var(--radius);
}

.lang-label {
  font-size: 0.85rem;
  color: var(--text-muted);
  display: flex;
  align-items: center;
  gap: 6px;
  font-weight: 500;
}

.lang-select {
  width: 100%;
  padding: 10px 14px;
  background: var(--bg-ctrl);
  border: 1px solid rgba(249, 115, 22, 0.4);
  color: #fff;
  border-radius: var(--radius-sm);
  font-size: 1rem;
  font-weight: 600;
  outline: none;
  cursor: pointer;
  transition: border-color 0.2s, box-shadow 0.2s;
}

.lang-select:focus {
  border-color: #f97316;
  box-shadow: 0 0 0 3px rgba(249, 115, 22, 0.25);
}

.header-right {
  display: flex;
  align-items: center;
  gap: 10px;
}

.lang-select-call {
  background: var(--bg-ctrl);
  border: 1px solid rgba(255, 255, 255, 0.15);
  color: #fff;
  padding: 4px 8px;
  border-radius: 8px;
  font-size: 0.8rem;
  font-weight: 600;
  cursor: pointer;
  outline: none;
}

.ended-avatar-wrap {
  width: 100px;
  height: 100px;
  border-radius: 50%;
  padding: 3px;
  background: linear-gradient(135deg, #f97316, #fbbf24);
  box-shadow: 0 6px 20px rgba(0, 0, 0, 0.4);
}

.ended-avatar-img {
  width: 100%;
  height: 100%;
  border-radius: 50%;
  object-fit: cover;
  display: block;
}

.headphone-hint {
  font-size: 0.85rem;
  color: var(--text-muted);
  background: var(--bg-card);
  border-radius: var(--radius-sm);
  padding: 8px 14px;
  text-align: center;
}

.btn-call-start {
  display: flex;
  align-items: center;
  gap: 10px;
  background: var(--green);
  color: #fff;
  border: none;
  border-radius: 50px;
  padding: 18px 44px;
  font-size: 1.25rem;
  font-weight: 700;
  cursor: pointer;
  box-shadow: 0 4px 24px rgba(34,197,94,0.35);
  transition: transform 0.1s, box-shadow 0.1s;
}

.btn-call-start:active {
  transform: scale(0.96);
  box-shadow: 0 2px 10px rgba(34,197,94,0.2);
}

.call-icon {
  font-size: 1.4rem;
}

.mic-error {
  color: #ff6060;
  font-size: 0.9rem;
  text-align: center;
  background: rgba(224,48,48,0.12);
  border-radius: var(--radius-sm);
  padding: 10px 16px;
}

/* ── Call screen ── */
#screen-call {
  justify-content: flex-start;
  gap: 14px;
  padding-top: 24px;
  padding-bottom: 120px;
  overflow-y: auto;
  -webkit-overflow-scrolling: touch;
  scroll-behavior: smooth;
  width: 100%;
}

#screen-call::-webkit-scrollbar {
  width: 4px;
}

#screen-call::-webkit-scrollbar-thumb {
  background: rgba(255, 255, 255, 0.15);
  border-radius: 4px;
}

.call-header {
  width: 100%;
  display: flex;
  align-items: center;
  justify-content: space-between;
}

.contact-info {
  display: flex;
  align-items: center;
  gap: 12px;
}

.contact-avatar {
  width: 52px;
  height: 52px;
  background: var(--bg-ctrl);
  border-radius: 50%;
  overflow: hidden;
  display: flex;
  align-items: center;
  justify-content: center;
  border: 2px solid #f97316;
  box-shadow: 0 2px 8px rgba(249, 115, 22, 0.3);
}

.contact-avatar-img {
  width: 100%;
  height: 100%;
  object-fit: cover;
  border-radius: 50%;
  display: block;
}

.contact-name {
  font-size: 1.1rem;
  font-weight: 700;
}

.call-status {
  font-size: 0.85rem;
  color: var(--text-muted);
}

.call-status.listening {
  color: var(--green);
}

.call-status.thinking {
  color: var(--accent);
}

.call-status.speaking {
  color: #f0a020;
}

.call-timer {
  font-size: 1.05rem;
  font-variant-numeric: tabular-nums;
  color: var(--text-muted);
  letter-spacing: 1px;
}

/* ── Captions ── */
.captions-area {
  width: 100%;
  background: var(--bg-card);
  border-radius: var(--radius);
  padding: 14px 16px;
  display: flex;
  flex-direction: column;
  gap: 8px;
  min-height: 110px;
  max-height: 38vh;
  overflow-y: auto;
  -webkit-overflow-scrolling: touch;
  scroll-behavior: smooth;
  line-height: 1.6;
  word-break: break-word;
}

.captions-area::-webkit-scrollbar {
  width: 4px;
}

.captions-area::-webkit-scrollbar-thumb {
  background: rgba(255, 255, 255, 0.2);
  border-radius: 4px;
}

.caption-row {
  display: flex;
  gap: 6px;
  align-items: flex-start;
  font-size: 1rem;
  line-height: 1.5;
}

.caption-label {
  font-weight: 700;
  color: var(--accent);
  flex-shrink: 0;
  min-width: 60px;
}

/* ── Service card (Clean Visit Button with Multilingual Prompt) ── */
.service-card {
  width: 100%;
  display: flex;
  flex-direction: column;
  justify-content: center;
  align-items: center;
  gap: 8px;
  padding: 8px 12px;
  text-align: center;
}

.service-card-prompt {
  font-size: 0.85rem;
  color: var(--text-muted);
  margin: 0;
  font-weight: 500;
  line-height: 1.4;
}

.service-card.hidden {
  display: none !important;
}

.service-site-btn {
  display: inline-flex;
  align-items: center;
  justify-content: center;
  gap: 10px;
  background: linear-gradient(135deg, #f97316, #ea580c);
  color: #fff;
  border-radius: 30px;
  padding: 12px 28px;
  font-size: 1.05rem;
  font-weight: 700;
  text-decoration: none;
  box-shadow: 0 4px 18px rgba(249, 115, 22, 0.4);
  transition: transform 0.15s, box-shadow 0.15s;
}

.service-site-btn:hover {
  transform: translateY(-2px);
  box-shadow: 0 6px 22px rgba(249, 115, 22, 0.55);
}

/* ── Call controls ── */
.call-controls {
  position: fixed;
  bottom: 0;
  left: 50%;
  transform: translateX(-50%);
  width: 100%;
  max-width: 480px;
  padding: 16px 20px 32px;
  background: linear-gradient(to top, var(--bg) 70%, transparent);
  display: flex;
  justify-content: center;
  gap: 20px;
}

.ctrl-btn {
  display: flex;
  flex-direction: column;
  align-items: center;
  gap: 4px;
  background: var(--bg-ctrl);
  border: none;
  border-radius: 50%;
  width: 68px;
  height: 68px;
  font-size: 0.72rem;
  color: var(--text);
  cursor: pointer;
  transition: transform 0.1s, background 0.15s;
}

.ctrl-btn span:first-child {
  font-size: 1.5rem;
}

.ctrl-btn:active {
  transform: scale(0.93);
}

.mute-btn.muted {
  background: var(--accent-dark);
}

.end-btn {
  background: var(--danger);
  color: #fff;
  box-shadow: 0 4px 18px rgba(224,48,48,0.4);
}

.end-btn:hover {
  background: var(--danger-dark);
}

/* ── Ended screen ── */
.ended-icon {
  font-size: 60px;
}

#screen-ended h2 {
  font-size: 1.5rem;
}

#screen-ended p {
  color: var(--text-muted);
}

/* ── Utilities ── */
.hidden {
  display: none !important;
}

'''

JS_CONTENT = r'''/**
 * call.js – Main browser call logic.
 *
 * Responsibilities:
 *  - Request mic permission (echoCancellation + noiseSuppression)
 *  - Load AudioWorklet, capture PCM16 at 16 kHz
 *  - Connect WebSocket to /ws/call
 *  - Play ringing sound, then connect
 *  - Forward mic PCM to server as binary frames
 *  - Receive JSON control messages and binary audio chunks
 *  - Manage playback queue; flush on stop_audio (barge-in)
 *  - Render call UI: state, captions, service card, timer, mute, end
 *
 * No external libraries, no build step.
 */

'use strict';

// ── State ──────────────────────────────────────────────────────────────────
let ws = null;
let audioCtx = null;
let workletNode = null;
let mediaStream = null;
let isMuted = false;
let isCaptionsOn = true;
let callStartTime = null;
let timerInterval = null;
let isEnded = false;
let currentLanguage = 'ta-IN';

const UI_TEXTS = {
  'ta-IN': {
    visitPrompt: 'அதிகாரப்பூர்வ பக்கத்தைப் பார்வையிட கீழே உள்ள பொத்தானைக் கிளிக் செய்யவும்',
    visitBtn: '🌐 அதிகாரப்பூர்வ இணையதளம்',
    thanksEnded: 'வள்ளியைத் தொடர்பு கொண்டதற்கு நன்றி! மேலும் விவரங்கள் தேவைப்பட்டால் என்னைத் தொடர்பு கொள்ளவும்.',
  },
  'hi-IN': {
    visitPrompt: 'आधिकारिक पृष्ठ पर जाने के लिए नीचे दिए गए बटन पर क्लिक करें',
    visitBtn: '🌐 आधिकारिक वेबसाइट',
    thanksEnded: 'वल्ली से संपर्क करने के लिए धन्यवाद! यदि आपको कोई और विवरण चाहिए तो मुझसे संपर्क करें।',
  },
  'te-IN': {
    visitPrompt: 'అధికారిక పేజీని సందర్శించడానికి క్రింది బటన్‌ను క్లిక్ చేయండి',
    visitBtn: '🌐 అధికారిక వెబ్‌సైట్',
    thanksEnded: 'వల్లిని సంప్రదించినందుకు ధన్యవాదాలు! మీకు ఏవైనా మరిన్ని వివరాలు కావాలంటే నన్ను సంప్రదించండి.',
  },
  'ml-IN': {
    visitPrompt: 'ഔദ്യോഗിക പേജ് സന്ദർശിക്കാൻ താഴെയുള്ള ബട്ടണിൽ ക്ലിക്കുചെയ്യുക',
    visitBtn: '🌐 ഔദ്യോഗിക വെബ്സൈറ്റ്',
    thanksEnded: 'വള്ളിയെ ബന്ധപ്പെട്ടതിന് നന്ദി! കൂടുതൽ വിവരങ്ങൾ ആവശ്യമുണ്ടെങ്കിൽ എന്നെ ബന്ധപ്പെടുക.',
  },
  'kn-IN': {
    visitPrompt: 'ಅಧಿಕೃತ ಪುಟಕ್ಕೆ ಭೇಟಿ ನೀಡಲು ಕೆಳಗಿನ ಬಟನ್ ಕ್ಲಿಕ್ ಮಾಡಿ',
    visitBtn: '🌐 ಅಧಿಕೃತ ವೆಬ್‌ಸೈಟ್',
    thanksEnded: 'ವಳ್ಳಿಯನ್ನು ಸಂಪರ್ಕಿಸಿದ್ದಕ್ಕಾಗಿ ಧನ್ಯವಾದಗಳು! ಹೆಚ್ಚಿನ ವಿವರಗಳು ಬೇಕಾದರೆ ನನ್ನನ್ನು ಸಂಪರ್ಕಿಸಿ.',
  },
};

function onLanguageChange(lang) {
  currentLanguage = lang;
  const sel1 = document.getElementById('lang-select');
  const sel2 = document.getElementById('call-lang-select');
  if (sel1) sel1.value = lang;
  if (sel2) sel2.value = lang;

  const t = UI_TEXTS[lang] || UI_TEXTS['ta-IN'];

  // Update ended screen message according to selected language
  const endedMsg = document.getElementById('ended-thanks-msg');
  if (endedMsg) {
    endedMsg.textContent = t.thanksEnded;
  }

  // Update service card texts if active
  const promptEl = document.getElementById('service-card-prompt');
  const siteBtn = document.getElementById('service-site-btn');
  if (promptEl) promptEl.textContent = t.visitPrompt;
  if (siteBtn) siteBtn.textContent = t.visitBtn;

  // Notify server if call WebSocket is active
  if (ws && ws.readyState === WebSocket.OPEN) {
    ws.send(JSON.stringify({ type: 'set_language', language: lang }));
  }
}

// Playback queue
let playbackQueue = [];
let isPlaying = false;
let hasReceivedAudioEnd = false;
let pendingServerState = null;
let pendingAudioMeta = null; // {format, rate, text}
let currentTurnAgentText = '';

// Energy VAD: track recent energy for barge-in hint
let recentEnergy = 0;
const ENERGY_THRESHOLD = 0.015; // tune to mic level

// ── UI helpers ──────────────────────────────────────────────────────────────
function showScreen(id) {
  document.querySelectorAll('.screen').forEach(el => {
    el.classList.remove('active');
    el.classList.add('hidden');
  });
  const s = document.getElementById(id);
  s.classList.remove('hidden');
  s.classList.add('active');
}

function setCallStatus(text, cls) {
  const el = document.getElementById('call-status');
  el.textContent = text;
  el.className = 'call-status' + (cls ? ' ' + cls : '');
}

function startTimer() {
  callStartTime = Date.now();
  timerInterval = setInterval(() => {
    const secs = Math.floor((Date.now() - callStartTime) / 1000);
    const m = Math.floor(secs / 60);
    const s = String(secs % 60).padStart(2, '0');
    document.getElementById('call-timer').textContent = `${m}:${s}`;
  }, 1000);
}

function stopTimer() {
  clearInterval(timerInterval);
  timerInterval = null;
}

// ── Ringtone ───────────────────────────────────────────────────────────────
function playRing() {
  const ring = document.getElementById('ring-audio');
  if (ring) {
    ring.volume = 0.5;
    ring.play().catch(() => {});
  }
}

function stopRing() {
  const ring = document.getElementById('ring-audio');
  if (ring) {
    ring.pause();
    ring.currentTime = 0;
  }
}

let currentSourceNode = null;

// ── Audio playback queue ────────────────────────────────────────────────────
async function enqueueAudio(arrayBuffer, sampleRate, text) {
  if (!audioCtx) return;
  playbackQueue.push({ arrayBuffer, sampleRate, text: text || '' });
  if (!isPlaying) playNextChunk();
}

function appendAgentCaption(text) {
  if (!isCaptionsOn || !text) return;
  if (currentTurnAgentText.length > 0) {
    currentTurnAgentText += ' ' + text;
  } else {
    currentTurnAgentText = text;
  }
  document.getElementById('agent-caption-text').textContent = currentTurnAgentText;
  const area = document.getElementById('captions-area');
  if (area) {
    area.scrollTop = area.scrollHeight;
  }
}

async function playNextChunk() {
  if (playbackQueue.length === 0) {
    isPlaying = false;
    currentSourceNode = null;
    pendingServerState = null;
    // When playback queue is empty, the agent has finished speaking -> immediately switch to Listening
    setCallStatus('Listening…', 'listening');
    if (hasReceivedAudioEnd) {
      sendPlaybackDone();
    }
    return;
  }
  isPlaying = true;
  setCallStatus('Speaking…', 'speaking');
  const { arrayBuffer, sampleRate, text } = playbackQueue.shift();

  // Display text in real time matching exactly what is currently being spoken
  if (text) {
    appendAgentCaption(text);
  }

  try {
    // Decode WAV
    const decoded = await audioCtx.decodeAudioData(arrayBuffer.slice(0));
    const src = audioCtx.createBufferSource();
    src.buffer = decoded;
    src.connect(audioCtx.destination);
    currentSourceNode = src;

    let endedHandled = false;
    const onEnded = () => {
      if (endedHandled) return;
      endedHandled = true;
      currentSourceNode = null;
      playNextChunk();
    };

    src.onended = onEnded;

    // Safety fallback timer: guarantee transition even if browser onended stalls
    if (decoded && decoded.duration) {
      setTimeout(onEnded, Math.round(decoded.duration * 1000) + 120);
    }

    src.start();
  } catch (e) {
    console.warn('Audio decode error:', e);
    playNextChunk();
  }
}

function sendPlaybackDone() {
  hasReceivedAudioEnd = false;
  if (ws && ws.readyState === WebSocket.OPEN) {
    ws.send(JSON.stringify({ type: 'playback_done' }));
  }
}

function flushPlayback() {
  playbackQueue = [];
  isPlaying = false;
  hasReceivedAudioEnd = false;
  pendingServerState = null;
  currentTurnAgentText = '';
  if (currentSourceNode) {
    try {
      currentSourceNode.stop();
    } catch (_) {}
    currentSourceNode = null;
  }
  setCallStatus('Listening…', 'listening');
  if (audioCtx && audioCtx.state === 'suspended') audioCtx.resume();
}

// ── Service card ────────────────────────────────────────────────────────────
function showServiceCard(entry) {
  const card = document.getElementById('service-card');
  const promptEl = document.getElementById('service-card-prompt');
  const siteBtn = document.getElementById('service-site-btn');
  if (!card || !siteBtn) return;

  const t = UI_TEXTS[currentLanguage] || UI_TEXTS['ta-IN'];
  if (promptEl) promptEl.textContent = t.visitPrompt;
  siteBtn.textContent = t.visitBtn;

  if (entry && entry.official_site) {
    siteBtn.href = entry.official_site;
    siteBtn.style.display = 'inline-flex';
    card.classList.remove('hidden');
  } else {
    card.classList.add('hidden');
  }
}

// ── Caption update ──────────────────────────────────────────────────────────
function updateCaption(who, text) {
  if (!isCaptionsOn) return;
  if (who === 'user') {
    const row = document.getElementById('user-caption');
    row.classList.remove('hidden');
    document.getElementById('user-caption-text').textContent = text;
  } else {
    document.getElementById('agent-caption-text').textContent = text;
  }
  const area = document.getElementById('captions-area');
  if (area) {
    area.scrollTop = area.scrollHeight;
  }
}

// ── WebSocket connection ────────────────────────────────────────────────────
function connectWS() {
  const proto = location.protocol === 'https:' ? 'wss:' : 'ws:';
  const url = `${proto}//${location.host}/ws/call`;
  ws = new WebSocket(url);
  ws.binaryType = 'arraybuffer';

  ws.onopen = () => {
    stopRing();
    setCallStatus('Connected', 'listening');
    ws.send(JSON.stringify({ type: 'start', language: currentLanguage }));
    startTimer();
  };

  ws.onmessage = (evt) => {
    if (typeof evt.data === 'string') {
      handleJSONMessage(JSON.parse(evt.data));
    } else {
      // Binary audio frame
      if (pendingAudioMeta) {
        enqueueAudio(evt.data, pendingAudioMeta.rate, pendingAudioMeta.text);
        pendingAudioMeta = null;
      }
    }
  };

  ws.onclose = () => {
    if (!isEnded) handleCallEnded();
  };

  ws.onerror = (e) => {
    console.error('WS error', e);
    setCallStatus('Connection error');
  };
}

function handleJSONMessage(msg) {
  switch (msg.type) {
    case 'state':
      if (msg.value === 'thinking') {
        currentTurnAgentText = '';
        document.getElementById('agent-caption-text').textContent = '…';
      }
      onStateChange(msg.value);
      break;
    case 'audio':
      pendingAudioMeta = { format: msg.format, rate: msg.rate, text: msg.text || '' };
      break;
    case 'audio_end':
      hasReceivedAudioEnd = true;
      if (!isPlaying && playbackQueue.length === 0) {
        setCallStatus('Listening…', 'listening');
        sendPlaybackDone();
      }
      break;
    case 'stop_audio':
      flushPlayback();
      break;
    case 'user_text':
      currentTurnAgentText = '';
      if (msg.final) updateCaption('user', msg.text);
      else updateCaption('user', msg.text + ' …');
      break;
    case 'agent_text':
      if (!isPlaying && !currentTurnAgentText) {
        updateCaption('agent', msg.text);
      }
      break;
    case 'service':
      showServiceCard(msg.entry);
      break;
    case 'error':
      console.error('Server error:', msg.message);
      setCallStatus('Error: ' + msg.message);
      break;
  }
}

function onStateChange(state) {
  pendingServerState = state;
  // If server indicates 'listening', but browser is still playing queued speech,
  // hold in Speaking… mode until playback finishes.
  if (state === 'listening' && (isPlaying || playbackQueue.length > 0)) {
    setCallStatus('Speaking…', 'speaking');
    return;
  }
  pendingServerState = null;
  const labels = {
    idle: 'Idle',
    listening: 'Listening…',
    thinking: 'Thinking…',
    speaking: 'Speaking…',
    ended: 'Call Ended',
  };
  setCallStatus(labels[state] || state, state);
  if (state === 'ended') handleCallEnded();
}

// ── Mic + AudioWorklet setup ────────────────────────────────────────────────
async function setupMic() {
  mediaStream = await navigator.mediaDevices.getUserMedia({
    audio: {
      echoCancellation: true,
      noiseSuppression: true,
      sampleRate: 44100,
    },
    video: false,
  });

  audioCtx = new AudioContext({ sampleRate: 44100 });
  await audioCtx.audioWorklet.addModule('/worklet.js');

  const source = audioCtx.createMediaStreamSource(mediaStream);
  workletNode = new AudioWorkletNode(audioCtx, 'pcm-capture', {
    processorOptions: { targetRate: 16000 },
  });
  source.connect(workletNode);
  // Don't connect workletNode to destination – we don't want loopback

  workletNode.port.onmessage = (e) => {
    const { type, buffer, value } = e.data;
    if (type === 'pcm') {
      // Do not stream microphone audio while speech is playing
      if (ws && ws.readyState === WebSocket.OPEN && !isMuted && !isPlaying && playbackQueue.length === 0) {
        ws.send(buffer);
      }
    } else if (type === 'energy') {
      recentEnergy = value;
    }
  };
}

// ── Public call actions ─────────────────────────────────────────────────────
async function startCall() {
  const errEl = document.getElementById('mic-error');
  errEl.classList.add('hidden');

  try {
    await setupMic();
  } catch (err) {
    errEl.textContent = '🎙️ Microphone access denied. Please allow mic access and try again.';
    errEl.classList.remove('hidden');
    return;
  }

  showScreen('screen-call');
  setCallStatus('Connecting…');
  playRing();

  // Small delay to let the ring play before WS connects
  setTimeout(connectWS, 800);
}

function toggleMute() {
  isMuted = !isMuted;
  const btn = document.getElementById('btn-mute');
  const icon = document.getElementById('mute-icon');
  if (isMuted) {
    btn.classList.add('muted');
    icon.textContent = '🔇';
    if (ws && ws.readyState === WebSocket.OPEN) {
      ws.send(JSON.stringify({ type: 'mute', on: true }));
    }
  } else {
    btn.classList.remove('muted');
    icon.textContent = '🎙️';
    if (ws && ws.readyState === WebSocket.OPEN) {
      ws.send(JSON.stringify({ type: 'mute', on: false }));
    }
  }
}

function toggleCaptions() {
  isCaptionsOn = !isCaptionsOn;
  const area = document.getElementById('captions-area');
  area.style.opacity = isCaptionsOn ? '1' : '0.2';
  document.getElementById('btn-captions').style.opacity = isCaptionsOn ? '1' : '0.5';
}

function endCall() {
  if (ws && ws.readyState === WebSocket.OPEN) {
    ws.send(JSON.stringify({ type: 'end' }));
    ws.close();
  }
  handleCallEnded();
}

function handleCallEnded() {
  if (isEnded) return;
  isEnded = true;
  stopTimer();
  stopRing();
  flushPlayback();

  if (workletNode) {
    workletNode.port.postMessage({ type: 'stop' });
    workletNode.disconnect();
    workletNode = null;
  }
  if (mediaStream) {
    mediaStream.getTracks().forEach(t => t.stop());
    mediaStream = null;
  }
  if (audioCtx) {
    audioCtx.close();
    audioCtx = null;
  }

  setTimeout(() => showScreen('screen-ended'), 300);
}

function resetUI() {
  isEnded = false;
  isMuted = false;
  isCaptionsOn = true;
  ws = null;
  pendingAudioMeta = null;
  playbackQueue = [];
  isPlaying = false;
  hasReceivedAudioEnd = false;
  pendingServerState = null;
  currentTurnAgentText = '';

  document.getElementById('service-card').classList.add('hidden');
  document.getElementById('user-caption').classList.add('hidden');
  document.getElementById('agent-caption-text').textContent = '…';
  document.getElementById('call-timer').textContent = '0:00';

  showScreen('screen-idle');
}

'''

WORKLET_CONTENT = r'''/**
 * worklet.js – AudioWorklet processor that captures PCM16 mono at 16 kHz.
 *
 * The worklet receives float32 samples at the browser's native sample rate.
 * It downsamples to 16 kHz, converts to PCM16 (little-endian int16), and
 * posts chunks of ~20 ms (320 samples at 16 kHz) to the main thread.
 *
 * Also performs a simple energy-based VAD: posts {type:"energy",value:N}
 * messages so the main thread can hint at barge-in.
 */
class PcmCapture extends AudioWorkletProcessor {
  constructor(options) {
    super();
    // Target sample rate (must match server: 16 kHz)
    this._targetRate = (options.processorOptions || {}).targetRate || 16000;
    // Resample ratio: how many input frames per output frame
    this._ratio = sampleRate / this._targetRate; // sampleRate = global in worklet
    this._accumulator = 0;
    this._outBuf = [];
    // Chunk size: 20 ms at 16 kHz = 320 samples
    this._chunkSize = Math.round(this._targetRate * 0.02);
    this._active = true;

    this.port.onmessage = (e) => {
      if (e.data && e.data.type === 'stop') {
        this._active = false;
      }
    };
  }

  process(inputs) {
    if (!this._active) return false;

    const channel = inputs[0] && inputs[0][0];
    if (!channel || channel.length === 0) return true;

    // Compute energy for VAD hint
    let sumSq = 0;
    for (let i = 0; i < channel.length; i++) sumSq += channel[i] * channel[i];
    const rms = Math.sqrt(sumSq / channel.length);
    this.port.postMessage({ type: 'energy', value: rms });

    // Nearest-neighbour downsampling + PCM16 conversion
    for (let i = 0; i < channel.length; i++) {
      this._accumulator += 1;
      if (this._accumulator >= this._ratio) {
        this._accumulator -= this._ratio;
        // Clamp and convert float32 to int16
        const s = Math.max(-1, Math.min(1, channel[i]));
        this._outBuf.push(Math.round(s < 0 ? s * 32768 : s * 32767));
      }

      if (this._outBuf.length >= this._chunkSize) {
        const pcm = new Int16Array(this._outBuf.splice(0, this._chunkSize));
        this.port.postMessage({ type: 'pcm', buffer: pcm.buffer }, [pcm.buffer]);
      }
    }

    return true;
  }
}

registerProcessor('pcm-capture', PcmCapture);

'''

print("✅ Web assets (Multilingual HTML, CSS, JS, AudioWorklet) loaded.")



In [ ]:
# [Cell 11] Main Application + ngrok Public Mobile URL + Phone QR Code
import asyncio
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import HTMLResponse, Response, FileResponse
import os
from pyngrok import ngrok
import qrcode
from IPython.display import display
import uvicorn

app = FastAPI(title="Valli - Government Services Voice Agent")

@app.on_event("startup")
async def startup():
    await preload_fillers()

@app.get("/", response_class=HTMLResponse)
async def index():
    return HTML_CONTENT

@app.get("/style.css")
async def style():
    return Response(content=CSS_CONTENT, media_type="text/css")

@app.get("/audio/ring.mp3")
async def ring():
    return Response(status_code=204)

@app.get("/valli.jpg")
async def valli_avatar():
    for p in ["valli.jpg", "govt-call-agent/static/valli.jpg", "static/valli.jpg"]:
        if os.path.exists(p):
            return FileResponse(p, media_type="image/jpeg")
    return Response(status_code=404)

@app.get("/call.js")
async def call_script():
    return Response(content=JS_CONTENT, media_type="text/javascript")

@app.get("/worklet.js")
async def worklet_script():
    return Response(content=WORKLET_CONTENT, media_type="text/javascript")

@app.get("/api/health")
async def health():
    return {"status": "ok", "assistant": "Valli"}

@app.websocket("/ws/call")
async def ws_call(ws: WebSocket):
    await ws.accept()
    session = FullCallSession(ws)
    try:
        await session.run()
    except WebSocketDisconnect:
        pass
    except Exception as exc:
        print("Session error:", exc)
    finally:
        try:
            await ws.close()
        except Exception:
            pass

# Connect ngrok (kill previous tunnel to prevent collisions on re-run)
ngrok.set_auth_token(NGROK_AUTHTOKEN)
try:
    ngrok.kill()
except Exception:
    pass

tunnel = ngrok.connect(8000, bind_tls=True)
public_url = tunnel.public_url

print("\n" + "=" * 65)
print("🚀 VALLI CALL AGENT IS LIVE!")
print("=" * 65)
print(f"\n🌐 Public URL for your phone: {public_url}\n")
print("📱 Instructions:")
print("1. Scan the QR code below on your phone (or open the URL in Chrome / Safari).")
print("2. If ngrok displays a warning page, tap 'Visit Site'.")
print("3. Tap 'Call Now' and grant microphone permissions.")
print("4. Speak with Valli in Tamil or Tanglish!")
print("=" * 65 + "\n")

# Display QR code for instant phone scanning
qr = qrcode.QRCode(box_size=6, border=2)
qr.add_data(public_url)
qr.make(fit=True)
img = qr.make_image(fill_color="black", back_color="white")
display(img)

# Start web app using the active Jupyter event loop (avoids asyncio.run conflict)
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

try:
    await server.serve()
except (KeyboardInterrupt, asyncio.CancelledError):
    print("\nShutting down Valli agent and closing ngrok...")
    ngrok.kill()


